In [1]:
# import libraries
import os
import random
import tensorflow as tf
import cv2
import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# seeding
SEED=42
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# load data
from google.colab import drive
path="/content/drive/MyDrive/Fundus images/dataset"
os.listdir(path)
#count the number of images
normal_dir=os.path.join(path,"normal")
cataract_dir=os.path.join(path,"cataract")
normal_count=len(os.listdir(normal_dir))
cataract_count=len(os.listdir(cataract_dir))
print("normal_count:",normal_count)
print("cataract_count:",cataract_count)
img_size=224
x=[]  # images
y=[]  # labels
# load normal images
for img in os.listdir(normal_dir):
  img_path=os.path.join(normal_dir,img)
  img_arr=cv2.imread(img_path)
  img_arr=cv2.cvtColor(img_arr,cv2.COLOR_BGR2RGB)
  img_arr=cv2.resize(img_arr,(img_size,img_size))
  x.append(img_arr)   #image
  y.append(0)   #label
# load cataract images
for img in os.listdir(cataract_dir):
  img_path=os.path.join(cataract_dir,img)
  img_arr=cv2.imread(img_path)
  img_arr=cv2.cvtColor(img_arr,cv2.COLOR_BGR2RGB)
  img_arr=cv2.resize(img_arr,(img_size,img_size))
  x.append(img_arr)
  y.append(1)
x=np.array(x)
y=np.array(y)
x=preprocess_input(x)
print(x.shape)
print(y.shape)
# dataset splitting
from sklearn.model_selection import train_test_split
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.4, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

train_gen=ImageDataGenerator(rotation_range=15,width_shift_range=0.1,height_shift_range=0.1,zoom_range=0.2,shear_range= 0.1,horizontal_flip=True, brightness_range= [0.8,1.2], fill_mode="nearest")
val_gen=ImageDataGenerator()

from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights=dict(enumerate(class_weights))
print(class_weights)

base_model=MobileNetV2(weights="imagenet",include_top=False,input_shape=(img_size,img_size,3))
base_model.trainable=False
x= base_model.output
x= GlobalAveragePooling2D()(x)
x= Dense(128,activation="relu")(x)
x= Dropout(0.5)(x)
output=Dense(1,activation="sigmoid")(x)
model=Model(inputs=base_model.input,outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4),loss="binary_crossentropy",metrics=["accuracy"])
model.summary()

history=model.fit(train_gen.flow(x_train,y_train,batch_size=32),epochs=15,validation_data=val_gen.flow(x_val,y_val),class_weight=class_weights)
base_model.trainable=True
for layer in base_model.layers[:-30]:
  layer.trainable=False
model.compile(optimizer=Adam(learning_rate=1e-5),loss="binary_crossentropy",metrics=["accuracy"])
history=model.fit(train_gen.flow(x_train,y_train,batch_size=32),epochs=15,validation_data=val_gen.flow(x_val,y_val),class_weight=class_weights)

loss,accuracy=model.evaluate(x_test,y_test)
print("loss:",loss)
print("accuracy:",accuracy)
from sklearn.metrics import classification_report,confusion_matrix
y_pred=model.predict(x_test)
y_pred=np.round(y_pred)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

normal_count: 1074
cataract_count: 1038
(2112, 224, 224, 3)
(2112,)
(1267, 224, 224, 3)
(422, 224, 224, 3)
(423, 224, 224, 3)
(1267,)
(422,)
(423,)
{0: np.float64(0.9836956521739131), 1: np.float64(1.0168539325842696)}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 111s 3s/step - accuracy: 0.7869 - loss: 0.4597 - val_accuracy: 0.9313 - val_loss: 0.2110
Epoch 2/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 107s 3s/step - accuracy: 0.9353 - loss: 0.2011 - val_accuracy: 0.9360 - val_loss: 0.1586
Epoch 3/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.9321 - loss: 0.1892 - val_accuracy: 0.9408 - val_loss: 0.1388
Epoch 4/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.9432 - loss: 0.1630 - val_accuracy: 0.9479 - val_loss: 0.1309
Epoch 5/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 93s 2s/step - accuracy: 0.9471 - loss: 0.1513 - val_accuracy: 0.9502 - val_loss: 0.1175
Epoch 6/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.9574 - loss: 0.1281 - val_accuracy: 0.9550 - val_loss: 0.1200
Epoch 7/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 91s 2s/step - accuracy: 0.9526 - loss: 0.1303 - val_accuracy: 0.9597 - val_loss: 0.1097
Epoch 8/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.9590 - loss: 0.1217 - val_accuracy: 0.9621 - val_lo

In [2]:
model.save("model.h5")

In [3]:
print("Last conv layer:", base_model.layers[-1].name)

Last conv layer: out_relu


In [4]:
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
img_size= 224
st.title ("Cataract Detection App")
@st.cache_resource
def load_my_model():
    model = load_model('model.h5')
    return model
model = load_my_model()
upload_file= st.file_uploader("Upload Image", type=['jpg','png','jpeg'])
if upload_file is not None:
    file_bytes = np.asarray(bytearray(upload_file.read()), dtype=np.uint8)
    img= cv2.imdecode(file_bytes,1)
    img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
    st.image(img, caption="Uploaded image", width= 300)
    img= cv2.resize(img,(224,224),interpolation=cv2.INTER_AREA)
    img= np.array(img)
    img= preprocess_input(img)
    img= np.expand_dims(img, axis=0)
    prediction= model.predict(img)[0][0]
    label = "Cataract" if prediction >= 0.5 else "Normal"
    confidence = prediction if prediction >= 0.5 else 1 - prediction
    st.write(f"**Result:** {label}")
    st.write(f"**Confidence:** {confidence * 100:.2f}%")
    st.progress(int(confidence * 100))

Writing app.py


In [6]:
! pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 38.9 MB/s eta 0:00:00


In [7]:
from pyngrok import ngrok
import os
ngrok.set_auth_token("39QKFLSxqL8Xk5czcatxvDLobXX_3dxoxRDoJB7H5kv58nqpW")
!pkill streamlit
get_ipython().system_raw("streamlit run app.py &")
public_url= ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://rushier-robbin-fractional.ngrok-free.dev" -> "http://localhost:8501"
